# IPP – Practical File 2 (Assignment 3)

**Name:** ____________________  
**Roll Number:** ____________________  
**Course:** Image Processing (IPP)

This notebook contains code for six image-processing experiments. Code cells are provided without line-by-line explanations. Short observations are printed where the problem statement asks for analysis.

## Setup
Install dependencies if needed:

```bash
pip install opencv-python numpy matplotlib scikit-image scikit-learn scipy
```

Place the input images in the notebook's working directory:
- Problem 1: `outdoor.jpg`
- Problem 2: `clean_gray.jpg`
- Problem 3: `smooth_gray.jpg`
- Problem 4: `jpeg_test.jpg`
- Problem 5: `texture_image.jpg`
- Problem 6: `plant_or_lesion.jpg`

Use grayscale images where specified. The notebook checks for missing files. You may replace filenames with your own images.


## Problem 1: Comparative Edge Detection on Natural Images

Apply Sobel, Laplacian and Canny edge detection to an outdoor natural image, display the outputs and print a short comparison.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

image = cv2.imread("outdoor.jpg")

if image is None:
    print("Image not found. Add an outdoor image named outdoor.jpg.")
else:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel = cv2.magnitude(sobel_x, sobel_y)
    sobel = cv2.convertScaleAbs(sobel)

    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian = cv2.convertScaleAbs(laplacian)

    canny = cv2.Canny(gray, 100, 200)

    titles = ["Original", "Sobel", "Laplacian", "Canny"]
    outputs = [cv2.cvtColor(image, cv2.COLOR_BGR2RGB), sobel, laplacian, canny]

    plt.figure(figsize=(14, 5))
    for i, (title, output) in enumerate(zip(titles, outputs), 1):
        plt.subplot(1, 4, i)
        plt.imshow(output, cmap=None if i == 1 else "gray")
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

    print("Short analysis:")
    print("- Sobel highlights edges with direction and may produce thicker responses.")
    print("- Laplacian responds to rapid intensity changes in all directions, but is sensitive to noise.")
    print("- Canny uses smoothing, non-maximum suppression and hysteresis thresholding, often producing thin, connected and meaningful edges.")
    print("For many natural scenes, Canny gives the clearest structural edges; the best result still depends on image content and thresholds.")

## Problem 2: Noise Sensitivity Experiment

Add Gaussian and salt-and-pepper noise to a grayscale image. Apply Sobel and Laplacian to both noisy images and compare edge clarity before and after Gaussian-blur preprocessing.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

gray = cv2.imread("clean_gray.jpg", cv2.IMREAD_GRAYSCALE)

if gray is None:
    print("Image not found. Add a grayscale image named clean_gray.jpg.")
else:
    rng = np.random.default_rng(42)

    # Gaussian noise
    gaussian_noise = rng.normal(0, 25, gray.shape)
    gaussian_noisy = np.clip(gray.astype(np.float32) + gaussian_noise, 0, 255).astype(np.uint8)

    # Salt-and-pepper noise
    sp_noisy = gray.copy()
    amount = 0.04
    random_map = rng.random(gray.shape)
    sp_noisy[random_map < amount / 2] = 0
    sp_noisy[random_map > 1 - amount / 2] = 255

    def edge_maps(image):
        sx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
        sy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
        sobel = cv2.convertScaleAbs(cv2.magnitude(sx, sy))
        lap = cv2.convertScaleAbs(cv2.Laplacian(image, cv2.CV_64F))
        return sobel, lap

    gaussian_sobel, gaussian_lap = edge_maps(gaussian_noisy)
    sp_sobel, sp_lap = edge_maps(sp_noisy)

    # Preprocessing: Gaussian blur for Gaussian noise; median filter for impulse noise
    gaussian_denoised = cv2.GaussianBlur(gaussian_noisy, (5, 5), 1.2)
    sp_denoised = cv2.medianBlur(sp_noisy, 5)

    gs_sobel, gs_lap = edge_maps(gaussian_denoised)
    ss_sobel, ss_lap = edge_maps(sp_denoised)

    images = [
        gray, gaussian_noisy, gaussian_sobel, gaussian_lap,
        sp_noisy, sp_sobel, sp_lap,
        gaussian_denoised, gs_sobel, gs_lap,
        sp_denoised, ss_sobel, ss_lap
    ]
    names = [
        "Clean", "Gaussian noisy", "Sobel: Gaussian", "Laplacian: Gaussian",
        "Salt & pepper", "Sobel: S&P", "Laplacian: S&P",
        "Gaussian blur", "Sobel after blur", "Laplacian after blur",
        "Median filter", "Sobel after median", "Laplacian after median"
    ]

    plt.figure(figsize=(16, 12))
    for i, (name, image) in enumerate(zip(names, images), 1):
        plt.subplot(4, 4, i)
        plt.imshow(image, cmap="gray")
        plt.title(name, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

    print("Observations:")
    print("- Laplacian is generally more affected by noise because it uses a second-order derivative.")
    print("- Sobel is also affected, but its first-order derivative and local averaging make it comparatively less noise-sensitive.")
    print("- Noise creates extra false edges and makes real edges less clear.")
    print("- Gaussian blur is useful for Gaussian noise; median filtering is especially effective for salt-and-pepper noise.")
    print("- Compare the displayed maps to judge the effect on this particular image.")

## Problem 3: Compression Ratio Comparison (RLE vs Huffman)

Run-Length Encoding and Huffman Coding are applied to a grayscale image. The bit lengths are calculated from the raw 8-bit pixels and from the encoded representations.

In [ ]:
import cv2
import numpy as np
import heapq
from collections import Counter

gray = cv2.imread("smooth_gray.jpg", cv2.IMREAD_GRAYSCALE)

if gray is None:
    print("Image not found. Add a grayscale image named smooth_gray.jpg.")
else:
    pixels = gray.flatten().tolist()
    original_bits = len(pixels) * 8

    # Run-Length Encoding: each run stores (pixel value, run length).
    # For a simple fixed-width estimate, value uses 8 bits and run length
    # uses enough bits to represent the longest possible run.
    runs = []
    current = pixels[0]
    count = 1
    for value in pixels[1:]:
        if value == current:
            count += 1
        else:
            runs.append((current, count))
            current, count = value, 1
    runs.append((current, count))

    run_length_bits = max(1, int(np.ceil(np.log2(len(pixels) + 1))))
    rle_bits = len(runs) * (8 + run_length_bits)

    # Huffman coding: build a frequency tree and calculate total code length.
    frequencies = Counter(pixels)
    heap = [[freq, [symbol, ""]] for symbol, freq in frequencies.items()]
    heapq.heapify(heap)

    if len(heap) == 1:
        huffman_code_lengths = {heap[0][1][0]: 1}
    else:
        while len(heap) > 1:
            lo = heapq.heappop(heap)
            hi = heapq.heappop(heap)
            for pair in lo[1:]:
                pair[1] = "0" + pair[1]
            for pair in hi[1:]:
                pair[1] = "1" + pair[1]
            heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
        huffman_code_lengths = {symbol: len(code) for symbol, code in heap[0][1:]}

    huffman_data_bits = sum(frequencies[symbol] * length
                            for symbol, length in huffman_code_lengths.items())
    # Include a simple 16-bit frequency per present symbol as a model-cost estimate.
    huffman_header_bits = len(frequencies) * (8 + 16)
    huffman_total_bits = huffman_data_bits + huffman_header_bits

    def report(name, compressed_bits):
        reduction = (1 - compressed_bits / original_bits) * 100
        ratio = original_bits / compressed_bits if compressed_bits else float("inf")
        print(f"{name}:")
        print(f"  Bit-length after compression (including estimated header): {compressed_bits}")
        print(f"  Compression ratio (original/compressed): {ratio:.3f}:1")
        print(f"  Percentage reduction: {reduction:.2f}%")

    print("Original bit-length:", original_bits)
    print("RLE bit-length (including fixed-width run records):", rle_bits)
    print("Huffman data bit-length:", huffman_data_bits)
    print("Huffman estimated header bit-length:", huffman_header_bits)
    report("RLE", rle_bits)
    report("Huffman", huffman_total_bits)

    print("Analysis:")
    print("- RLE works well when adjacent pixels repeat in long runs, such as large uniform regions.")
    print("- RLE can be inefficient when pixel values change frequently.")
    print("- Huffman coding assigns shorter codes to frequent values and longer codes to rare values.")
    print("- The header/codebook overhead affects small images and images with many distinct values.")
    print("- These are educational estimates; practical file formats also include metadata and implementation-specific overhead.")

## Problem 4: JPEG Block Artifact Analysis (8×8 DCT + Quantisation)

Apply an 8×8 block-based JPEG-style transform with low, medium and high quantisation. Compare reconstructed images using PSNR and SSIM.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

image = cv2.imread("jpeg_test.jpg", cv2.IMREAD_GRAYSCALE)

if image is None:
    print("Image not found. Add an image named jpeg_test.jpg.")
else:
    # Standard JPEG luminance quantisation matrix
    Q50 = np.array([
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99]
    ], dtype=np.float32)

    # Pad image dimensions to multiples of 8
    h, w = image.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    padded = np.pad(image, ((0, pad_h), (0, pad_w)), mode="edge").astype(np.float32)

    def jpeg_style_reconstruct(gray_image, q_scale):
        result = np.zeros_like(gray_image, dtype=np.float32)
        quant_matrix = np.maximum(1, Q50 * q_scale)

        for y in range(0, gray_image.shape[0], 8):
            for x in range(0, gray_image.shape[1], 8):
                block = gray_image[y:y+8, x:x+8] - 128
                dct_block = cv2.dct(block)
                quantized = np.round(dct_block / quant_matrix)
                dequantized = quantized * quant_matrix
                reconstructed = cv2.idct(dequantized) + 128
                result[y:y+8, x:x+8] = reconstructed

        return np.clip(result[:h, :w], 0, 255).astype(np.uint8)

    levels = {
        "Low quantisation": 0.5,
        "Medium quantisation": 1.0,
        "High quantisation": 2.0
    }

    reconstructions = {}
    print("Quality metrics (higher PSNR/SSIM generally indicates better fidelity):")
    for label, scale in levels.items():
        reconstructed = jpeg_style_reconstruct(padded, scale)
        reconstructions[label] = reconstructed
        psnr = peak_signal_noise_ratio(image, reconstructed, data_range=255)
        ssim = structural_similarity(image, reconstructed, data_range=255)
        print(f"{label}: PSNR = {psnr:.2f} dB, SSIM = {ssim:.4f}")

    display_images = [image] + list(reconstructions.values())
    display_titles = ["Original"] + list(reconstructions.keys())

    plt.figure(figsize=(14, 5))
    for i, (title, result) in enumerate(zip(display_titles, display_images), 1):
        plt.subplot(1, 4, i)
        plt.imshow(result, cmap="gray", vmin=0, vmax=255)
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

    print("Analysis:")
    print("- Block artifacts may appear as visible 8×8 boundaries, especially in smooth areas and around sharp transitions.")
    print("- Fine textures and small details are represented by high-frequency DCT coefficients; stronger quantisation removes more of them.")
    print("- Increasing quantisation strength generally lowers PSNR and SSIM because more image information is discarded.")
    print("- Actual metric values depend on the input image and quantisation level.")

## Problem 5: Texture-Based Segmentation Using GLCM Features

Calculate contrast, energy and homogeneity from local image neighbourhoods. Cluster the feature vectors into three groups using K-means and display the segmentation.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import graycomatrix, graycoprops
from sklearn.cluster import KMeans
from scipy.ndimage import zoom

image = cv2.imread("texture_image.jpg", cv2.IMREAD_GRAYSCALE)

if image is None:
    print("Image not found. Add a texture image named texture_image.jpg.")
else:
    # Reduce grey levels to make GLCM calculation manageable
    quantized = (image // 16).astype(np.uint8)

    window_size = 16
    stride = 8
    feature_rows = []
    positions = []

    for y in range(0, image.shape[0] - window_size + 1, stride):
        for x in range(0, image.shape[1] - window_size + 1, stride):
            patch = quantized[y:y+window_size, x:x+window_size]
            glcm = graycomatrix(
                patch,
                distances=[1],
                angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                levels=16,
                symmetric=True,
                normed=True
            )

            contrast = graycoprops(glcm, "contrast").mean()
            energy = graycoprops(glcm, "energy").mean()
            homogeneity = graycoprops(glcm, "homogeneity").mean()

            feature_rows.append([contrast, energy, homogeneity])
            positions.append((y, x))

    features = np.asarray(feature_rows, dtype=np.float32)

    if len(features) < 3:
        print("Image is too small for three clusters with the selected window size.")
    else:
        # Standardise features before clustering
        mean = features.mean(axis=0)
        std = features.std(axis=0)
        std[std == 0] = 1
        standard_features = (features - mean) / std

        kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
        labels = kmeans.fit_predict(standard_features)

        # Convert patch labels to a coarse segmentation map
        rows = len(range(0, image.shape[0] - window_size + 1, stride))
        cols = len(range(0, image.shape[1] - window_size + 1, stride))
        label_grid = labels.reshape(rows, cols)
        segmentation = zoom(
            label_grid,
            (image.shape[0] / rows, image.shape[1] / cols),
            order=0
        )[:image.shape[0], :image.shape[1]]

        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(image, cmap="gray")
        plt.title("Original Image")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(segmentation, cmap="viridis", vmin=0, vmax=2)
        plt.title("GLCM + K-means Segmentation (k=3)")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

        print("Feature observations:")
        print("- Contrast measures local intensity variation and helps distinguish rough or strongly varying textures.")
        print("- Energy measures textural uniformity; it is often higher in more regular or uniform regions.")
        print("- Homogeneity is higher when neighbouring grey levels are similar.")
        print("- Combining all three features can separate textures better than relying on one feature alone.")
        print("- Similar-looking textures, lighting changes and the selected window size can cause mixed or incorrect clusters.")
        print("Cluster labels 0, 1 and 2 are arbitrary and do not represent fixed texture names.")

## Problem 6: Multi-Feature Segmentation for Disease Detection

Create a feature vector using grayscale intensity, Sobel gradient magnitude and local texture variance. Apply K-means clustering and display the segmented regions. The cluster labels are exploratory and must be interpreted against the actual input image.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

image = cv2.imread("plant_or_lesion.jpg")

if image is None:
    print("Image not found. Add a plant or skin-lesion image named plant_or_lesion.jpg.")
else:
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.float32)

    # Feature 1: intensity
    intensity = gray / 255.0

    # Feature 2: Sobel gradient magnitude
    grad_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    gradient = cv2.magnitude(grad_x, grad_y)
    gradient = gradient / (gradient.max() + 1e-8)

    # Feature 3: local texture variance
    local_mean = cv2.blur(gray, (9, 9))
    local_mean_sq = cv2.blur(gray ** 2, (9, 9))
    local_variance = np.maximum(local_mean_sq - local_mean ** 2, 0)
    local_variance = local_variance / (local_variance.max() + 1e-8)

    # Build a three-feature vector for every pixel
    feature_stack = np.stack([intensity, gradient, local_variance], axis=-1)
    pixel_features = feature_stack.reshape(-1, 3)

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(pixel_features)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_features)
    segmented = labels.reshape(gray.shape)

    # Display each feature and the clustering result
    plt.figure(figsize=(14, 8))

    plt.subplot(2, 3, 1)
    plt.imshow(rgb)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(2, 3, 2)
    plt.imshow(intensity, cmap="gray")
    plt.title("Intensity")
    plt.axis("off")

    plt.subplot(2, 3, 3)
    plt.imshow(gradient, cmap="gray")
    plt.title("Sobel Gradient Magnitude")
    plt.axis("off")

    plt.subplot(2, 3, 4)
    plt.imshow(local_variance, cmap="gray")
    plt.title("Local Texture Variance")
    plt.axis("off")

    plt.subplot(2, 3, 5)
    plt.imshow(segmented, cmap="viridis")
    plt.title("K-means Segmentation (k=3)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # Compare the relative variation of each feature across clusters
    cluster_means = np.array([
        feature_stack[segmented == cluster].mean(axis=0)
        for cluster in range(3)
    ])

    print("Mean feature values by cluster [intensity, gradient, local variance]:")
    for cluster, values in enumerate(cluster_means):
        print(f"Cluster {cluster}: {values}")

    print("Evaluation:")
    print("- Intensity separates regions with different brightness or colour-derived grayscale levels.")
    print("- Gradient magnitude highlights boundaries and abrupt changes.")
    print("- Local variance highlights regions with different texture or surface irregularity.")
    print("- The most useful feature depends on the image: inspect cluster means and feature maps rather than assuming one feature always dominates.")
    print("- Misclassification can occur due to shadows, highlights, background pixels, natural texture variation or overlap between healthy and diseased appearances.")
    print("- K-means labels are arbitrary. Confirm which cluster corresponds to disease using visual inspection or labelled ground-truth data.")
    print("- This is an image-segmentation exercise, not a validated medical or plant-disease diagnosis.")